In [1]:
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader
import librosa
import numpy as np
import pandas as pd
import os
from tqdm import tqdm
import matplotlib.pyplot as plt
import warnings
from sklearn.preprocessing import LabelEncoder
from sklearn.model_selection import train_test_split
warnings.filterwarnings('ignore')

In [2]:
print(f'PyTorch: {torch.__version__}')
print(f'CUDA available: {torch.cuda.is_available()}')
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Using device: {device}')

PyTorch: 2.5.1+cu121
CUDA available: True
Using device: cuda


In [3]:
df = pd.read_csv(r"C:\Users\Arup Sarkar\Documents\EmoWave\data\dataemotion_data.csv")

N_MELS = 128
SR = 22050
DURATION = 3
N_FFT = 2048
HOP_LENGTH = 512
NUM_CLASSES = 8

print(f'Total clips: {len(df)}')
print(f'Classes: {df["emotion"].nunique()}')

Total clips: 1440
Classes: 8


In [8]:
le = LabelEncoder()
df['label'] = le.fit_transform(df['emotion'])

class EmoWaveDataset(Dataset):
    def __init__(self, df):
        self.df = df.reset_index(drop=True)

    def __len__(self):
        return len(self.df)

    def __getitem__(self, idx):
        row = self.df.iloc[idx]
        file_path = row['path']

        y, sr = librosa.load(file_path, sr=SR, duration=DURATION)

        if len(y) < SR * DURATION:
            y = np.pad(y, (0, SR * DURATION - len(y)))

        S = librosa.feature.melspectrogram(y=y, sr=sr, n_mels=N_MELS,
                                           n_fft=N_FFT, hop_length=HOP_LENGTH)
        S_db = librosa.power_to_db(S, ref=np.max)

        S_db = (S_db - S_db.mean()) / (S_db.std() + 1e-8)

        tensor = torch.tensor(S_db, dtype=torch.float32).unsqueeze(0)

        label = torch.tensor(row['label'], dtype=torch.long)
        return tensor, label

print('Dataset class defined')

Dataset class defined


In [9]:
train_df, test_df = train_test_split(
    df, test_size=0.2, random_state=42, stratify=df['label']
)

train_dataset = EmoWaveDataset(train_df)
test_dataset = EmoWaveDataset(test_df)

train_loader = DataLoader(train_dataset, batch_size=32, shuffle=True, num_workers=0)
test_loader = DataLoader(test_dataset, batch_size=32, shuffle=False, num_workers=0)

print(f'Train samples: {len(train_dataset)}')
print(f'Test samples:  {len(test_dataset)}')
print(f'Train batches: {len(train_loader)}')
print(f'Test batches:  {len(test_loader)}')

# verify one batch
batch_x, batch_y = next(iter(train_loader))
print(f'Batch X shape: {batch_x.shape}')
print(f'Batch y shape: {batch_y.shape}')

Train samples: 1152
Test samples:  288
Train batches: 36
Test batches:  9
Batch X shape: torch.Size([32, 1, 128, 130])
Batch y shape: torch.Size([32])
